In [12]:
import numpy as np
import pandas as pd
from sklearn.metrics import precision_score, recall_score, mean_squared_error
from scipy.sparse.linalg import svds
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

In [13]:
def load_data(file_path):
    df = pd.read_csv(file_path)
    print(f"Loaded {df.shape[0]} customer records")
    return df

In [14]:
def prepare_data(df):
    # Handle missing numerical features values 
    df['Booking_Frequency'] = df['Booking_Frequency'].fillna(df['Booking_Frequency'].median())
    df['Avg_Spending'] = df['Avg_Spending'].fillna(df['Avg_Spending'].median())

    # Handle missing categorical features values
    df['Preferred_Service'] = df['Preferred_Service'].fillna(df['Preferred_Service'].mode()[0])
    
    services = df['Preferred_Service'].unique()
    
    user_item_matrix = pd.DataFrame(0, index=df['Customer_ID'], columns=services)
    
    for idx, row in df.iterrows():
        user_item_matrix.at[row['Customer_ID'], row['Preferred_Service']] = 1
    
    return user_item_matrix

In [15]:
def train_collaborative_filtering(user_item_matrix, n_factors=5):
    matrix = user_item_matrix.values
    
    user_ratings_mean = np.mean(matrix, axis=1)
    
    matrix_normalized = matrix - user_ratings_mean.reshape(-1, 1)
    
    n_factors = min(n_factors, min(matrix.shape) - 1)
    u, sigma, vt = svds(matrix_normalized, k=n_factors)
    
    sigma = np.diag(sigma)
    
    user_features = u
    item_features = vt.T
    
    model = {
        'user_features': user_features,
        'item_features': item_features,
        'sigma': sigma,
        'user_ratings_mean': user_ratings_mean,
        'services': user_item_matrix.columns,
        'user_ids': user_item_matrix.index
    }
    
    predictions = np.dot(np.dot(u, sigma), vt) + user_ratings_mean.reshape(-1, 1)
    predicted_matrix = pd.DataFrame(
        predictions, 
        index=user_item_matrix.index, 
        columns=user_item_matrix.columns
    )
    
    return model, predicted_matrix

In [16]:
def evaluate_model(actual_matrix, predicted_matrix, threshold=0.5):
    actual_binary = (actual_matrix.values > 0).astype(int)
    predicted_binary = (predicted_matrix.values > threshold).astype(int)
    
    mse = mean_squared_error(actual_matrix.values.flatten(), predicted_matrix.values.flatten())
    
    precision_scores = []
    recall_scores = []
    
    for i in range(len(actual_binary)):
        if np.sum(predicted_binary[i]) > 0:
            try:
                precision_scores.append(precision_score(actual_binary[i], predicted_binary[i]))
            except:
                precision_scores.append(0)
        else:
            precision_scores.append(0)
        
        if np.sum(actual_binary[i]) > 0:
            try:
                recall_scores.append(recall_score(actual_binary[i], predicted_binary[i]))
            except:
                recall_scores.append(0)
        else:
            recall_scores.append(0)
    
    avg_precision = np.mean(precision_scores)
    avg_recall = np.mean(recall_scores)
    
    return {
        'mse': mse,
        'precision': avg_precision,
        'recall': avg_recall
    }

In [17]:
def recommend_services(user_id, predicted_matrix, n=3):
    if user_id not in predicted_matrix.index:
        print(f"User {user_id} not found in the model.")
        return []
    
    user_predictions = predicted_matrix.loc[user_id]
    
    recommended_services = user_predictions.sort_values(ascending=False).head(n)
    
    return recommended_services

In [18]:
def visualize_results(user_item_matrix, predicted_matrix):
    max_users = min(8, len(user_item_matrix.index))
    max_services = min(5, len(user_item_matrix.columns))
    
    sample_users = np.random.choice(user_item_matrix.index, size=max_users, replace=False)
    sample_services = np.random.choice(user_item_matrix.columns, size=max_services, replace=False)
    
    actual_sample = user_item_matrix.loc[sample_users, sample_services]
    predicted_sample = predicted_matrix.loc[sample_users, sample_services]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    sns.heatmap(actual_sample, ax=ax1, cmap="YlGnBu", vmin=0, vmax=1)
    ax1.set_title("Actual Service Usage")
    ax1.set_xlabel("Services")
    ax1.set_ylabel("Customers")
    
    sns.heatmap(predicted_sample, ax=ax2, cmap="YlGnBu", vmin=0, vmax=1)
    ax2.set_title("Predicted Service Ratings")
    ax2.set_xlabel("Services")
    ax2.set_ylabel("Customers")
    
    plt.tight_layout()
    plt.savefig("collaborative_filtering_visualization.png")
    plt.close()
    
    return fig

In [19]:
def save_model(model, predicted_matrix, user_item_matrix, output_file="recommendation_model.pkl"):
    model_package = {
        'cf_model': model,
        'predicted_matrix': predicted_matrix,
        'user_item_matrix': user_item_matrix
    }
    
    joblib.dump(model_package, output_file)
    print(f"Model saved to {output_file}")

In [21]:
def main():
    df = load_data('/Users/biratpoudel/Desktop/blys/customer_data_100.csv')
    
    user_item_matrix = prepare_data(df)
    
    n_factors = min(3, min(user_item_matrix.shape) - 1)
    cf_model, predicted_matrix = train_collaborative_filtering(user_item_matrix, n_factors=n_factors)
    
    metrics = evaluate_model(user_item_matrix, predicted_matrix)
    print(f"Model Performance:")
    print(f"  MSE: {metrics['mse']:.4f}")
    print(f"  Precision: {metrics['precision']:.4f}")
    print(f"  Recall: {metrics['recall']:.4f}")
    
    print("\nSample Recommendations:")
    for user_id in list(user_item_matrix.index)[:3]:  # First 3 users
        recommendations = recommend_services(user_id, predicted_matrix)
        print(f"\nUser {user_id}:")
        print(recommendations)
    
    visualize_results(user_item_matrix, predicted_matrix)
    
    save_model(cf_model, predicted_matrix, user_item_matrix)
    
    return cf_model, predicted_matrix, user_item_matrix

In [22]:
if __name__ == "__main__":
    main()

Loaded 100 customer records
Model Performance:
  MSE: 0.0487
  Precision: 0.5600
  Recall: 0.5600

Sample Recommendations:

User 1001:
Manicure        0.416794
Massage         0.362650
Yoga Session    0.239274
Name: 1001, dtype: float64

User 1002:
Facial              0.998703
Yoga Session        0.031732
Wellness Package   -0.001105
Name: 1002, dtype: float64

User 1003:
Wellness Package    0.999058
Yoga Session        0.027041
Facial             -0.001105
Name: 1003, dtype: float64
Model saved to recommendation_model.pkl
